In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import joblib
import xgboost as xgb

In [2]:
df_train = pd.read_csv('merge/result/I1_전처리ver05_AB오버샘플링.csv')
df_test = pd.read_parquet('merge/result/Segment_merge_test_ver_05.parquet')

In [3]:
# 특성과 타겟 분리
X_train = df_train.drop(columns=['Segment', 'ID'])
y_train = df_train['Segment']

# 라벨 인코딩
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

# 검증용 데이터 분할
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train_encoded, test_size=0.2, stratify=y_train_encoded, random_state=42)

In [4]:
# 5. 다중분류 모델 학습 (GPU + GridSearchCV)
params = {
    'max_depth': [4],
    'learning_rate': [0.1],
    'n_estimators': [100],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
}
xgb_multi = XGBClassifier(
    tree_method='hist',    
    device='cuda',         
    eval_metric='mlogloss',  
    verbosity=1
)

grid_multi = GridSearchCV(xgb_multi, param_grid=params, cv=3, scoring='f1_macro', verbose=1, n_jobs=1)
print("[다중분류 모델 학습 중...]")
grid_multi.fit(X_tr, y_tr)

[다중분류 모델 학습 중...]
Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\Lee\anaconda3\envs\tensorflow_env\lib\site-packages\xgboost\core.py:729: UserWarning: [15:17:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device='cuda',
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='mlogloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constra...
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=1,
             param_grid={'colsample_bytree': [0.8], 'learning_rate': [0.1],
                         'max_depth': [4], 'n_estimators': [100],
                         'subsample': [0.8]},
             scoring='f1_macro', verbose=1)

In [5]:
# A vs All 이진 분류 모델 학습
y_binary_A = (y_train == 'A').astype(int)
X_res_A, y_res_A = SMOTE(random_state=42).fit_resample(X_train, y_binary_A)
X_res_A = X_res_A.drop(columns=['ID'], errors='ignore')
model_A = XGBClassifier(tree_method = "hist", device = "cuda", eval_metric='logloss')
model_A.fit(X_res_A, y_res_A)

# B vs All 이진 분류 모델 학습
y_binary_B = (y_train == 'B').astype(int)
X_res_B, y_res_B = SMOTE(random_state=42).fit_resample(X_train, y_binary_B)
X_res_B = X_res_B.drop(columns=['ID'], errors='ignore')
model_B = XGBClassifier(tree_method= "hist", device = "cuda", eval_metric='logloss')
model_B.fit(X_res_B, y_res_B)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [6]:
# 검증 데이터 예측 및 보정
print("[검증 데이터에서 성능 평가 중...]")
y_val_pred_multi = grid_multi.predict(X_val)
y_val_pred_multi_label = le.inverse_transform(y_val_pred_multi)

A_val_proba = model_A.predict_proba(X_val)[:, 1]
B_val_proba = model_B.predict_proba(X_val)[:, 1]

threshold_A = 0.85
threshold_B = 0.85

corrected_val_pred_label = []
for i in range(len(y_val_pred_multi_label)):
    if A_val_proba[i] > threshold_A:
        corrected_val_pred_label.append('A')
    elif B_val_proba[i] > threshold_B:
        corrected_val_pred_label.append('B')
    else:
        corrected_val_pred_label.append(y_val_pred_multi_label[i])

[검증 데이터에서 성능 평가 중...]


In [7]:
# 성능 평가용 라벨 인코딩 복원
y_val_true_label = le.inverse_transform(y_val)
y_val_true_encoded = le.transform(y_val_true_label)
corrected_val_pred_encoded = le.transform(corrected_val_pred_label)

# 성능 평
print("[이진 분류 보정 후 검증 성능 평가 결과]")
print("Accuracy:", accuracy_score(y_val_true_encoded, corrected_val_pred_encoded))
print("Precision:", precision_score(y_val_true_encoded, corrected_val_pred_encoded, average='macro'))
print("Recall:", recall_score(y_val_true_encoded, corrected_val_pred_encoded, average='macro'))
print("F1 Score:", f1_score(y_val_true_encoded, corrected_val_pred_encoded, average='macro'))
print("\nClassification Report:\n", classification_report(y_val_true_encoded, corrected_val_pred_encoded, target_names=le.classes_))

[이진 분류 보정 후 검증 성능 평가 결과]
Accuracy: 0.8919357472554503
Precision: 0.8234337350634279
Recall: 0.8227530477266782
F1 Score: 0.8184548907139872

Classification Report:
               precision    recall  f1-score   support

           A       0.82      1.00      0.90      2000
           B       0.97      1.00      0.99      2000
           C       0.71      0.55      0.62     25518
           D       0.68      0.59      0.63     69848
           E       0.93      0.97      0.95    384411

    accuracy                           0.89    483777
   macro avg       0.82      0.82      0.82    483777
weighted avg       0.88      0.89      0.89    483777



In [8]:
# 테스트셋 ID 기준 그룹 평균 처리 및 예측
X_test = df_test.copy()
X_test_grouped = X_test.groupby('ID').mean(numeric_only=True).reset_index()

# 다중분류 모델 예측
y_test_pred_multi = grid_multi.predict(X_test_grouped.drop(columns=['ID']))
y_test_pred_multi_label = le.inverse_transform(y_test_pred_multi)

# 이진 분류 모델 확률 예측
A_proba_test = model_A.predict_proba(X_test_grouped.drop(columns=['ID']))[:, 1]
B_proba_test = model_B.predict_proba(X_test_grouped.drop(columns=['ID']))[:, 1]

In [9]:
# 보정 적용
def final_prediction(multi, a_prob, b_prob):
    if a_prob > threshold_A:
        return 'A'
    elif b_prob > threshold_B:
        return 'B'
    else:
        return multi

final_pred = [final_prediction(multi, a, b) for multi, a, b in zip(y_test_pred_multi_label, A_proba_test, B_proba_test)]

In [12]:
# 저장
submission = pd.DataFrame({
    'ID': X_test_grouped['ID'],
    'Predicted_Segment': final_pred
})
submission.to_csv('merge/result/J2_XGB_이진분류적용.csv', index=False)
print("최종 결과 파일 저장 완료: 'J2_XGB_이진분류적용.csv'")

최종 결과 파일 저장 완료: 'J2_XGB_이진분류적용.csv'
